In [ ]:
import dlt
from pyspark.sql import functions as F

# -------------------------------------------------------------------------
# CONFIGURATION
# Supported loc_type: 'geojson', 'socrata_json', 'lat_lon', or 'wkt'
# -------------------------------------------------------------------------
TABLE_CONFIGS = [
    {
        "bronze_table": "bronze_catalog.bronze_schema.bronze_sample_geojson",
        "silver_table_name": "silver_sample_geojson",
        "table_comment": "Silver table parsed from Socrata GeoJSON geometries with iMAP spatial columns.",
        "loc_type": "geojson",
        "geojson_col": "the_geom",         # Name of your GeoJSON column
        "source_srid": 4326,               # Lon/lat degrees default to 4326
        "drop_raw_loc_col": False          # Set to True to omit the raw JSON column
    },
    {
        "bronze_table": "bronze_catalog.bronze_schema.bronze_sample_socrata",
        "silver_table_name": "silver_sample_socrata",
        "table_comment": "Silver table with Socrata parsed addresses and iMAP spatial columns.",
        "loc_type": "socrata_json",
        "raw_loc_col": "location",
        "source_srid": 4326,
        "drop_raw_loc_col": False
    }
]

def build_silver_sql(cfg: dict) -> str:
    """Builds the transformation SQL matching iMAP silver table specifications."""
    bronze = cfg["bronze_table"]
    loc_type = cfg["loc_type"]
    srid = cfg.get("source_srid", 4326)

    # 1. Base attribute & coordinate unpacking
    if loc_type == "geojson":
        raw_col = cfg["geojson_col"]
        base_projection = "b.*"
        # Parse GeoJSON string safely into native Geometry with SRID
        g_expression = f"""
            CASE 
                WHEN {raw_col} IS NOT NULL AND length(trim({raw_col})) > 0
                THEN st_setsrid(st_geomfromgeojson({raw_col}), {srid})
                ELSE NULL 
            END
        """
    elif loc_type == "socrata_json":
        raw_col = cfg["raw_loc_col"]
        base_projection = f"""
            b.*,
            try_cast(get_json_object({raw_col}, '$.latitude') AS DOUBLE) AS latitude,
            try_cast(get_json_object({raw_col}, '$.longitude') AS DOUBLE) AS longitude,
            get_json_object(get_json_object({raw_col}, '$.human_address'), '$.address') AS street_address,
            get_json_object(get_json_object({raw_col}, '$.human_address'), '$.city') AS city,
            get_json_object(get_json_object({raw_col}, '$.human_address'), '$.state') AS state,
            get_json_object(get_json_object({raw_col}, '$.human_address'), '$.zip') AS zip
        """
        g_expression = f"""
            CASE 
                WHEN longitude BETWEEN -180 AND 180 
                 AND latitude BETWEEN -90 AND 90 
                 AND NOT (longitude = 0 AND latitude = 0)
                THEN st_point(longitude, latitude, {srid})
                ELSE NULL 
            END
        """
    elif loc_type == "lat_lon":
        lon_col = cfg["lon_col"]
        lat_col = cfg["lat_col"]
        base_projection = f"""
            b.*,
            try_cast({lon_col} AS DOUBLE) AS __lon,
            try_cast({lat_col} AS DOUBLE) AS __lat
        """
        g_expression = f"""
            CASE 
                WHEN __lon BETWEEN -180 AND 180 
                 AND __lat BETWEEN -90 AND 90 
                 AND NOT (__lon = 0 AND __lat = 0)
                THEN st_point(__lon, __lat, {srid})
                ELSE NULL 
            END
        """
    elif loc_type == "wkt":
        wkt_col = cfg["wkt_col"]
        base_projection = "b.*"
        g_expression = f"st_setsrid(try_to_geometry({wkt_col}), {srid})"
    else:
        raise ValueError(f"Unsupported loc_type: {loc_type}")

    # 2. Strict SRID literal transforms
    geom_expr = "__g" if srid == 4326 else "st_transform(__g, 4326)"
    geom_md_expr = "__g" if srid == 6487 else "st_transform(__g, 6487)"

    # 3. Clean up internal temporary columns
    omit_cols = ["__g"]
    if loc_type == "lat_lon":
        omit_cols.extend(["__lon", "__lat"])
    if cfg.get("drop_raw_loc_col"):
        col_to_drop = cfg["geojson_col"] if loc_type == "geojson" else cfg.get("raw_loc_col")
        if col_to_drop:
            omit_cols.append(col_to_drop)
            
    except_clause = f"EXCEPT({', '.join(omit_cols)})"

    # 4. Final SQL assembly: Attributes -> geom -> geom_md -> h3_r8
    return f"""
    WITH base_data AS (
        SELECT {base_projection}
        FROM {bronze} b
    ),
    with_base_geometry AS (
        SELECT *,
               {g_expression} AS __g
        FROM base_data
    ),
    with_transformed_geometry AS (
        SELECT *,
               {geom_expr} AS geom,
               {geom_md_expr} AS geom_md
        FROM with_base_geometry
    )
    SELECT * {except_clause},
           CASE 
               WHEN geom IS NULL OR st_isempty(geom) THEN NULL
               WHEN abs(st_x(st_centroid(geom))) > 180 OR abs(st_y(st_centroid(geom))) > 90 THEN NULL
               ELSE h3_longlatash3(st_x(st_centroid(geom)), st_y(st_centroid(geom)), 8)
           END AS h3_r8
    FROM with_transformed_geometry
    """

# -------------------------------------------------------------------------
# DECLARATIVE PIPELINE TABLE FACTORY
# -------------------------------------------------------------------------
def create_silver_dlt_table(cfg: dict):
    table_name = cfg["silver_table_name"]
    srid = cfg.get("source_srid", 4326)
    
    geom_comment = "Geometry in EPSG:4326 (source SRID)" if srid == 4326 else f"Geometry in EPSG:4326 (transformed from source SRID {srid})"
    geom_md_comment = "Geometry in EPSG:6487 (source SRID)" if srid == 6487 else f"Geometry in EPSG:6487 (transformed from source SRID {srid})"
    h3_comment = "H3 cell (resolution 8) of the geometry centroid in EPSG:4326"

    @dlt.table(
        name=table_name,
        comment=cfg.get("table_comment", "Silver spatial table"),
        table_properties={
            "delta.autoOptimize.optimizeWrite": "true",
            "delta.autoOptimize.autoCompact": "true",
            "delta.dataSkippingStatsColumns": "h3_r8"
        }
    )
    @dlt.expect("centroid_h3_integrity", "geom IS NULL OR st_isempty(geom) OR h3_r8 IS NOT NULL")
    def silver_table():
        query = build_silver_sql(cfg)
        return spark.sql(query)

# Register tables into the pipeline
for config in TABLE_CONFIGS:
    create_silver_dlt_table(config)